# CalibrateQwen 02: off-policy soft-target distillation
We train on fixed teacher completions while Tinker transfers the teacher's top token distribution at every supervised position.

In [ ]:
from pathlib import Path
REPO_ROOT = Path('/content/AutoRegressive-Bhasha')
if not REPO_ROOT.exists():
    !git clone https://github.com/ritwikraha/AutoRegressive-Bhasha.git /content/AutoRegressive-Bhasha
%cd /content/AutoRegressive-Bhasha/calibrate_qwen
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ['TINKER_API_KEY'] = userdata.get('TINKER_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
except Exception:
    pass

In [ ]:
from training.prepare_training_data import prepare_training_file
from training.train_off_policy import OffPolicyConfig, train_off_policy

conversation_file = 'artifacts/training/teacher_numeric.jsonl'
prepare_training_file(output_path=conversation_file, variant='teacher', confidence_format='numeric')
config = OffPolicyConfig(
    conversation_file=conversation_file,
    log_path='/content/calibrate_qwen_runs/off_policy_soft',
    model_name='Qwen/Qwen3.5-4B',
    teacher_model='Qwen/Qwen3.5-9B',
    batch_size=16,
    n_teacher_targets=20,
    teacher_concurrency=16,
    max_steps=None,  # Set 3 for a paid pipeline smoke test
    wandb_project='calibrate-qwen' if os.environ.get('WANDB_API_KEY') else None,
    wandb_name='off_policy_soft',
)
config

In [ ]:
await train_off_policy(config)